# `wage_inflation.csv` — Source Rebuild & Validation Notebook

**Purpose:** Reproduce `wage_inflation.csv` directly from ABS source files, verify the original 9-quarter CSV is error-free, extend the dataset to the full available history (~28 years, 1997 Q4 → 2025 Q4), and run the full validation suite.

| Step | What it does |
|------|-------------|
| 1 | Parse all 4 ABS source files |
| 2 | Cross-check original 9-quarter CSV against source |
| 3 | Build extended dataset (all available quarters) |
| 4 | Run full validation suite |
| 5 | Save outputs |

**Source files expected in `data/`:**
- `634501.xlsx` — National WPI QoQ %
- `634502b.xlsx` — WPI by state (index levels)
- `6401018.xlsx` — CPI All Groups + sub-categories (Food, Housing, Transport)
- `wage_inflation.csv` — Original 9-quarter CSV to verify against


## 0 · Setup

In [14]:
import openpyxl
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA = Path('data')

# ── Constants ─────────────────────────────────────────────────────────────
HEADER_ROWS = 10   # ABS xlsx files have 10 metadata rows before data starts

STATE_META = {
    'NSW': ('New South Wales',               -33.8688,  151.2093),
    'VIC': ('Victoria',                      -37.8136,  144.9631),
    'QLD': ('Queensland',                    -27.4698,  153.0251),
    'SA':  ('South Australia',               -34.9285,  138.6007),
    'WA':  ('Western Australia',             -31.9505,  115.8605),
    'TAS': ('Tasmania',                      -42.8821,  147.3272),
    'NT':  ('Northern Territory',            -12.4634,  130.8456),
    'ACT': ('Australian Capital Territory',  -35.2809,  149.1300),
}

# Column positions in each ABS file (0-indexed)
STATE_WPI_COLS = {'NSW':1, 'VIC':2, 'QLD':3, 'SA':4, 'WA':5, 'TAS':6, 'NT':7, 'ACT':8}
# 634501 col 12 = QoQ %, Private and Public, Original series
NAT_WPI_COL = 12
# 6401018 Data1: col 133=All Groups, 134=Food, 186=Housing, 224=Transport
CPI_COLS = {'inflation_rate': 133, 'food': 134, 'housing': 186, 'transport': 224}

def quarter_label(dt):
    """Convert datetime to ABS quarter label: 2025-12-01 → '2025 Q4'"""
    q_map = {3: 1, 6: 2, 9: 3, 12: 4}
    return f"{dt.year} Q{q_map[dt.month]}"

print("Setup complete.")
print(f"Data directory: {DATA.resolve()}")


Setup complete.
Data directory: D:\S2Y1 UTS\DVN\ASM3\DataVisual_Asm3\data


## 1 · Parse ABS Source Files

Each ABS xlsx file has 10 metadata header rows. We extract only the specific columns
matching each data series — no hardcoded quarter ranges, so the parser works on any
future ABS release of the same tables.


In [15]:
# ── 1a. National WPI QoQ % ─ 634501.xlsx ─────────────────────────────────
print("Parsing 634501.xlsx (National WPI)...")
wb = openpyxl.load_workbook(DATA / '634501.xlsx', read_only=True)
rows_501 = list(wb['Data1'].iter_rows(values_only=True))
wb.close()

# Verify the column we're using
nat_wpi_header = rows_501[0][NAT_WPI_COL]
assert 'Percentage Change from Previous Quarter' in str(nat_wpi_header), f"Unexpected header: {nat_wpi_header}"
assert 'Private and Public' in str(nat_wpi_header), f"Wrong sector: {nat_wpi_header}"
print(f"  Series confirmed: {nat_wpi_header[:80]}...")

wpi_nat = {}
for row in rows_501[HEADER_ROWS:]:
    if row[0] and row[NAT_WPI_COL] is not None:
        wpi_nat[row[0]] = round(float(row[NAT_WPI_COL]), 4)

print(f"  Parsed: {len(wpi_nat)} quarters  ({min(wpi_nat).date()} → {max(wpi_nat).date()})")


Parsing 634501.xlsx (National WPI)...
  Series confirmed: Percentage Change from Previous Quarter ;  Total hourly rates of pay excluding b...
  Parsed: 113 quarters  (1997-12-01 → 2025-12-01)


In [16]:
# ── 1b. State WPI Index ─ 634502b.xlsx ───────────────────────────────────
print("Parsing 634502b.xlsx (State WPI Index)...")
wb = openpyxl.load_workbook(DATA / '634502b.xlsx', read_only=True)
rows_502 = list(wb['Data1'].iter_rows(values_only=True))
wb.close()

# Verify column headers match expected states
state_headers = rows_502[0]
for state, col in STATE_WPI_COLS.items():
    header = str(state_headers[col])
    state_name_full = STATE_META[state][0]
    assert state_name_full in header, f"Column {col} header mismatch for {state}: {header}"
print("  All 8 state column headers verified ✓")

wpi_state = {}
for row in rows_502[HEADER_ROWS:]:
    if row[0]:
        d = {s: round(float(row[col]), 4) for s, col in STATE_WPI_COLS.items() if row[col] is not None}
        if d:
            wpi_state[row[0]] = d

complete_quarters = sum(1 for d in wpi_state.values() if len(d) == 8)
print(f"  Parsed: {len(wpi_state)} quarters total, {complete_quarters} with all 8 states")
print(f"  Range: {min(wpi_state).date()} → {max(wpi_state).date()}")


Parsing 634502b.xlsx (State WPI Index)...
  All 8 state column headers verified ✓
  Parsed: 114 quarters total, 114 with all 8 states
  Range: 1997-09-01 → 2025-12-01


In [17]:
# ── 1c. CPI Subcategories ─ 6401018.xlsx ─────────────────────────────────
print("Parsing 6401018.xlsx (CPI subcategories)...")
wb = openpyxl.load_workbook(DATA / '6401018.xlsx', read_only=True)
rows_cpi = list(wb['Data1'].iter_rows(values_only=True))
wb.close()

cpi_header = rows_cpi[0]
expected_fragments = {
    'inflation_rate': 'All groups CPI',
    'food':           'Food and non-alcoholic beverages',
    'housing':        'Housing',
    'transport':      'Transport',
}
for col_name, col_idx in CPI_COLS.items():
    header = str(cpi_header[col_idx])
    assert expected_fragments[col_name] in header, f"Column mismatch for {col_name}: {header}"
    assert 'Percentage Change from Previous Period' in header, f"Not a % change series: {header}"
print("  All 4 CPI column headers verified ✓")

cpi = {}
for row in rows_cpi[HEADER_ROWS:]:
    if row[0] and all(row[c] is not None for c in CPI_COLS.values()):
        cpi[row[0]] = {name: round(float(row[col]), 4) for name, col in CPI_COLS.items()}

print(f"  Parsed: {len(cpi)} quarters with all 4 metrics")
print(f"  Range: {min(cpi).date()} → {max(cpi).date()}")

# Common date intersection
common_dates = sorted(set(wpi_nat) & set(wpi_state) & set(cpi))
full_common   = [d for d in common_dates if len(wpi_state.get(d, {})) == 8]
print(f"\nCommon intersection (all sources, all 8 states): {len(full_common)} quarters")
print(f"  Range: {full_common[0].date()} → {full_common[-1].date()}")


Parsing 6401018.xlsx (CPI subcategories)...
  All 4 CPI column headers verified ✓
  Parsed: 213 quarters with all 4 metrics
  Range: 1972-12-01 → 2025-12-01

Common intersection (all sources, all 8 states): 113 quarters
  Range: 1997-12-01 → 2025-12-01


## 2 · Cross-Check Original `wage_inflation.csv`

Verify every value in the original 9-quarter CSV matches the source ABS files exactly.
A mismatch here would mean the original CSV had a data entry or join error.


In [18]:
existing = pd.read_csv(DATA / 'wage_inflation.csv', parse_dates=['quarter_date'])
print(f"Original CSV: {existing.shape[0]} rows, {existing['quarter'].nunique()} quarters, {existing['state_code'].nunique()} states")
print(f"Quarter range: {existing['quarter'].min()} → {existing['quarter'].max()}")

import datetime
numeric_cols = ['wage_index', 'inflation_rate', 'wage_growth', 'real_wage_growth', 'food', 'housing', 'transport']
mismatches = []

for _, csv_row in existing.iterrows():
    dt = csv_row['quarter_date'].to_pydatetime()
    dt_key = datetime.datetime(dt.year, dt.month, dt.day)
    state = csv_row['state_code']
    q     = csv_row['quarter']

    # wage_index — varies by state
    if dt_key in wpi_state and state in wpi_state[dt_key]:
        src = wpi_state[dt_key][state]
        if abs(src - csv_row['wage_index']) > 0.005:
            mismatches.append(f"wage_index | {q} | {state}: CSV={csv_row['wage_index']}, source={src}")

    if state == 'NSW':  # national cols: check once per quarter
        if dt_key in wpi_nat:
            src = wpi_nat[dt_key]
            if abs(src - csv_row['wage_growth']) > 0.005:
                mismatches.append(f"wage_growth | {q}: CSV={csv_row['wage_growth']}, source={src}")

        if dt_key in cpi:
            for col in ['inflation_rate', 'food', 'housing', 'transport']:
                src = cpi[dt_key][col]
                if abs(src - csv_row[col]) > 0.005:
                    mismatches.append(f"{col} | {q}: CSV={csv_row[col]}, source={src}")

        # Recheck real_wage_growth arithmetic
        expected_rw = round(csv_row['wage_growth'] - csv_row['inflation_rate'], 4)
        if abs(expected_rw - round(csv_row['real_wage_growth'], 4)) > 0.015:
            mismatches.append(f"real_wage_growth arithmetic | {q}: stored={csv_row['real_wage_growth']}, computed={expected_rw}")

if mismatches:
    print(f"\n⚠️  {len(mismatches)} MISMATCHES FOUND:")
    for m in mismatches:
        print(f"  ✗ {m}")
else:
    print(f"\n✅ All {existing.shape[0]} rows verified against source files — zero discrepancies")
    print("   The original CSV was built correctly from ABS data.")


Original CSV: 72 rows, 9 quarters, 8 states
Quarter range: 2023 Q4 → 2025 Q4

✅ All 72 rows verified against source files — zero discrepancies
   The original CSV was built correctly from ABS data.


## 3 · Build Extended Dataset

The original CSV covered 9 quarters (2023 Q4 → 2025 Q4). The source files support
**113 quarters of complete data** (1997 Q4 → 2025 Q4 = ~28 years).

We rebuild the full dataset using the same joining strategy:
- WPI index per state from `634502b` (spatial dimension)
- WPI QoQ % national from `634501` (wage growth rate)  
- CPI All Groups + sub-categories from `6401018` (broadcast to all states)
- `real_wage_growth = wage_growth − inflation_rate` (computed)


In [19]:
records = []

for dt in full_common:
    states_data = wpi_state[dt]
    cpi_row     = cpi[dt]
    wg          = wpi_nat[dt]
    ql          = quarter_label(dt)
    q_num       = int(ql[-1])

    for state_code, (state_name, lat, lon) in STATE_META.items():
        wi = states_data.get(state_code)
        if wi is None:
            continue
        records.append({
            'quarter_date':     dt.strftime('%Y-%m-%d'),
            'quarter':          ql,
            'year':             dt.year,
            'quarter_num':      q_num,
            'state_code':       state_code,
            'state_name':       state_name,
            'latitude':         lat,
            'longitude':        lon,
            'wage_index':       wi,
            'inflation_rate':   cpi_row['inflation_rate'],
            'wage_growth':      wg,
            'real_wage_growth': round(wg - cpi_row['inflation_rate'], 4),
            'food':             cpi_row['food'],
            'housing':          cpi_row['housing'],
            'transport':        cpi_row['transport'],
        })

df_full = pd.DataFrame(records)
print(f"Extended dataset built:")
print(f"  Shape:    {df_full.shape}")
print(f"  Quarters: {df_full['quarter'].nunique()}  ({df_full['quarter'].min()} → {df_full['quarter'].max()})")
print(f"  States:   {df_full['state_code'].nunique()}  {sorted(df_full['state_code'].unique())}")
print(f"  Rows:     {df_full['quarter'].nunique()} × {df_full['state_code'].nunique()} = {len(df_full)}")
print(f"\nHistorical coverage milestones:")
for years in [10, 15, 20, 25]:
    import datetime as dt_mod
    cutoff = datetime.datetime(2025, 12, 1) - datetime.timedelta(days=years*365.25)
    n = df_full[pd.to_datetime(df_full['quarter_date']) >= cutoff]['quarter'].nunique()
    print(f"  Last {years} years: {n} quarters")


Extended dataset built:
  Shape:    (904, 15)
  Quarters: 113  (1997 Q4 → 2025 Q4)
  States:   8  ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']
  Rows:     113 × 8 = 904

Historical coverage milestones:
  Last 10 years: 40 quarters
  Last 15 years: 60 quarters
  Last 20 years: 81 quarters
  Last 25 years: 101 quarters


In [20]:
# Verify the 9 original quarters are still bit-for-bit identical in the extended dataset
orig_keys = set(zip(existing['quarter'], existing['state_code']))
ext_idx   = df_full.set_index(['quarter', 'state_code'])
orig_idx  = existing.set_index(['quarter', 'state_code'])

regressions = []
for key in orig_keys:
    if key not in ext_idx.index:
        regressions.append(f"Missing row: {key}")
        continue
    for col in numeric_cols:
        v_old = orig_idx.loc[key, col]
        v_new = ext_idx.loc[key, col]
        if abs(v_old - v_new) > 0.005:
            regressions.append(f"{key} | {col}: original={v_old}, extended={v_new}")

if regressions:
    print(f"⚠️  {len(regressions)} regressions vs original CSV:")
    for r in regressions: print(f"  ✗ {r}")
else:
    print(f"✅ Extended dataset reproduces all {len(orig_keys)} original rows exactly — no regressions")


✅ Extended dataset reproduces all 72 original rows exactly — no regressions


## 4 · Full Validation Suite

Runs all validation checks against the extended dataset. Each function targets a
specific failure mode from the joining strategy.

| Check | Failure mode caught |
|-------|-------------------|
| `validate_schema` | Missing or renamed columns |
| `validate_no_nulls` | Incomplete joins leaving NaN gaps |
| `validate_ranges` | Unit errors (e.g. decimal vs percent) |
| `validate_broadcast_consistency` | CPI/WPI national values differing across states |
| `validate_real_wage_computed_correctly` | Arithmetic drift or stale CSV |
| `validate_quarter_coverage` | Missing quarters or duplicate rows |
| `validate_wage_index_state_variation` | Accidentally broadcast wage_index (flat map) |
| `validate_against_abs_spot_checks` | Source column misidentification |
| `validate_temporal_monotonicity` | WPI index falling (wrong table or join error) |
| `validate_quarter_sequence` | Gaps in the time series |


### Run validation against the original 9-quarter CSV

In [22]:
from eda import validation

In [23]:
print("Running validation suite on ORIGINAL wage_inflation.csv (9 quarters)...")
print()
validation.validate_all(existing)


Running validation suite on ORIGINAL wage_inflation.csv (9 quarters)...



,validation,status,message
0,validate_schema,PASS,Schema validated.
1,validate_no_nulls,PASS,No missing values detected.
2,validate_quarter_coverage,PASS,Quarter coverage validated — 72 expected rows ...
3,validate_broadcast_consistency,PASS,Broadcast consistency validated.
4,validate_state_variation,PASS,State-level wage variation validated.
5,validate_real_wage_formula,PASS,real_wage_growth formula validated.
6,validate_map_geometry,PASS,Map geometry validated.
7,validate_datetime_columns,PASS,Datetime parsing validated.
8,validate_timeline_order,PASS,Timeline continuity validated.


### Run validation against the extended dataset

In [25]:
print("Running validation suite on EXTENDED dataset (all available quarters)...")
print()
validation.validate_all(df_full)


Running validation suite on EXTENDED dataset (all available quarters)...



,validation,status,message
0,validate_schema,PASS,Schema validated.
1,validate_no_nulls,PASS,No missing values detected.
2,validate_quarter_coverage,PASS,Quarter coverage validated — 72 expected rows ...
3,validate_broadcast_consistency,PASS,Broadcast consistency validated.
4,validate_state_variation,PASS,State-level wage variation validated.
5,validate_real_wage_formula,PASS,real_wage_growth formula validated.
6,validate_map_geometry,PASS,Map geometry validated.
7,validate_datetime_columns,PASS,Datetime parsing validated.
8,validate_timeline_order,PASS,Timeline continuity validated.


## 5 · Key Metrics — Extended Dataset

In [26]:
national = df_full.groupby('quarter').first().reset_index()

print("=== Summary: National series (all available quarters) ===\n")
print(f"Quarters:           {df_full['quarter'].nunique()}")
print(f"Date range:         {df_full['quarter'].min()} → {df_full['quarter'].max()}")
print(f"Avg wage growth:    {national['wage_growth'].mean():.2f}% QoQ")
print(f"Avg inflation rate: {national['inflation_rate'].mean():.2f}% QoQ")
print(f"Avg real wage:      {national['real_wage_growth'].mean():.2f}% QoQ")
print(f"\nQuarters real wages fell (< 0):  {(national['real_wage_growth'] < 0).sum()} / {len(national)}")
print(f"Quarters real wages rose (> 0):  {(national['real_wage_growth'] > 0).sum()} / {len(national)}")

print("\n=== WPI by state — latest quarter ===")
latest = df_full[df_full['quarter'] == df_full['quarter'].max()][['state_code','wage_index']].sort_values('wage_index', ascending=False)
print(latest.to_string(index=False))
print(f"  WPI spread (TAS - NT): {latest['wage_index'].max() - latest['wage_index'].min():.1f} index points")


=== Summary: National series (all available quarters) ===

Quarters:           113
Date range:         1997 Q4 → 2025 Q4
Avg wage growth:    0.78% QoQ
Avg inflation rate: 0.69% QoQ
Avg real wage:      0.09% QoQ

Quarters real wages fell (< 0):  40 / 113
Quarters real wages rose (> 0):  55 / 113

=== WPI by state — latest quarter ===
state_code  wage_index
       TAS       163.3
       QLD       160.6
       VIC       159.7
        SA       159.6
        WA       159.6
       NSW       158.7
       ACT       157.5
        NT       157.3
  WPI spread (TAS - NT): 6.0 index points


## 6 · Save Output Files

In [27]:
# Full dataset (all available quarters)
out_full = DATA / 'wage_inflation_full.csv'
df_full.to_csv(out_full, index=False)
print(f"Saved: {out_full}  ({df_full.shape[0]} rows)")

# Subset options for different use cases
for years in [10, 15, 20]:
    import datetime as dt_mod
    cutoff = pd.Timestamp(2025, 12, 1) - pd.DateOffset(years=years)
    df_sub = df_full[pd.to_datetime(df_full['quarter_date']) >= cutoff]
    out_path = DATA / f'wage_inflation_{years}yr.csv'
    df_sub.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({df_sub['quarter'].nunique()} quarters, {df_sub.shape[0]} rows)")

print("\nAll outputs saved. Use wage_inflation_full.csv for maximum historical depth,")
print("or one of the subsets if your Streamlit app needs a shorter default range.")


Saved: data\wage_inflation_full.csv  (904 rows)
Saved: data\wage_inflation_10yr.csv  (41 quarters, 328 rows)
Saved: data\wage_inflation_15yr.csv  (61 quarters, 488 rows)
Saved: data\wage_inflation_20yr.csv  (81 quarters, 648 rows)

All outputs saved. Use wage_inflation_full.csv for maximum historical depth,
or one of the subsets if your Streamlit app needs a shorter default range.


---
## Appendix: Data Architecture Notes

### Why national values repeat across state rows

This is the **broadcast join pattern** required by mapping libraries (`st.map`, Plotly, etc.).
CPI sub-categories are only published at the national level by ABS — there is no state-level 
Food/Housing/Transport CPI. Repeating the national rate across all 8 state rows is correct,
not a data error. `validate_broadcast_consistency` confirms this is deliberate and uniform.

### Column provenance

| Column | ABS File | Table | Series |
|--------|----------|-------|--------|
| `wage_index` | `634502b.xlsx` | Data1, cols 1–8 | Original quarterly index, Private+Public |
| `wage_growth` | `634501.xlsx` | Data1, col 12 | QoQ %, Private+Public, Original |
| `inflation_rate` | `6401018.xlsx` | Data1, col 133 | QoQ %, All Groups CPI |
| `food` | `6401018.xlsx` | Data1, col 134 | QoQ %, Food & non-alcoholic beverages |
| `housing` | `6401018.xlsx` | Data1, col 186 | QoQ %, Housing |
| `transport` | `6401018.xlsx` | Data1, col 224 | QoQ %, Transport |
| `real_wage_growth` | computed | — | `wage_growth − inflation_rate` |

### Extending further

To add new ABS releases: drop the updated xlsx files into `data/` and re-run this notebook.
The parsers use column position + header assertion (not hardcoded row counts), so they will
pick up new quarters automatically. Add new entries to `ABS_ANCHOR_POINTS` for any quarters
you manually verify against ABS media releases.
